In [139]:
import matplotlib as mtp
import pandas as pd
import numpy as np

opay = pd.read_csv("../archive/olist_order_payments_dataset.csv") #
ordit = pd.read_csv("../archive/olist_order_items_dataset.csv") #
orev = pd.read_csv("../archive/olist_order_reviews_dataset.csv") #
osel = pd.read_csv("../archive/olist_sellers_dataset.csv") # 
oprod =  pd.read_csv("../archive/olist_products_dataset.csv") #

opay.drop_duplicates()
ordit.drop_duplicates()
orev.drop_duplicates()
osel.drop_duplicates()
oprod.drop_duplicates()

opay = opay.replace(r'^\s*$', np.nan, regex=True)
ordit = ordit.replace(r'^\s*$', np.nan, regex=True)
orev = orev.replace(r'^\s*$', np.nan, regex=True)
osel = osel.replace(r'^\s*$', np.nan, regex=True)
oprod = oprod.replace(r'^\s*$', np.nan, regex=True)

orev["review_comment_message"] = orev["review_comment_message"].replace(
    r"^\s*$", np.nan, regex=True
)
assert not orev["review_comment_message"].dropna().str.fullmatch(r"\s*").any()

In [140]:
# número de reviews com nota menor que 4
menor_q_4 = orev.loc[orev['review_score'] < 4]
print(f"{len(menor_q_4)} reviews com nota menor que 4")

distribuicao =(
    menor_q_4.groupby('review_score')
    .agg(
        qtd=('review_score','count')
    )
)
# porcentagem de reviews com nota menor que 4
media_menor_que_4 = len(orev.query("review_score <  4")) / len(orev) * 100
print(f"{media_menor_que_4:.1f}% menor que 4")

# média geral das reviews
media_geral = float(orev["review_score"].mean())
print(f"{media_geral:.2f}/5 - média geral das reviews")
# truncado = float(str(media)[: str(media).find(".") + 3])

condicao = (
    (orev["review_score"] < 4)
    & orev["review_comment_message"].notna()
    & orev["review_comment_message"].str.strip().ne("")
)

# below4comments = condicao.mean()
# below4comments

complaints = pd.DataFrame(orev.loc[condicao])
complaints = complaints.drop_duplicates()
complaints
complaints.to_csv("created-tables/complained_reviews.csv")
distribuicao


22754 reviews com nota menor que 4
22.9% menor que 4
4.09/5 - média geral das reviews


,qtd
review_score,
1,11424
2,3151
3,8179


Gerando um novo csv com os dados limpos que precisamos da tabela de reviews

In [141]:
df = (
    complaints[['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']]
    .merge(
        ordit[['order_id','order_item_id','product_id','seller_id','price','freight_value']],
        on='order_id',
        how='left'
    ).merge(
        osel[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
        on='seller_id',
        how='left'
    ).merge(
        oprod[['product_id','product_category_name']],
        on='product_id',
        how='left'
    ).merge(
        opay,
        on='order_id',
        how='left'
    )
)

df = df.drop_duplicates()
df['seller_zip_code_prefix'] = df['seller_zip_code_prefix'].astype(str).str.strip()  
df['order_item_id'] = df['order_item_id'].astype('Int64')
df['payment_sequential'] = df['payment_sequential'].astype('Int64')
df['payment_installments'] = df['payment_installments'].astype('Int64')
df.dropna(how="all")

df.shape
df.to_csv("created-tables/complained_reviews_full_data.csv")

Gerando uma nova tabela unindo dados de outras para uma melhor análise

In [142]:
# !!! PROBLEMA: Estou encontrando muitas reviews e orders repetidas.

# pd.unique(df['review_id'])
rep_an = df[
    [
        'review_id',
        'order_id',
        'price',
        'freight_value',
        'payment_value',
        'payment_type',
        'review_comment_message',
        'review_score',
    ]
].dropna(subset='price')
rep_an.to_csv('created-tables/repeated_values_analysis.csv')
rep_an

,review_id,order_id,price,freight_value,payment_value,payment_type,review_comment_message,review_score
0,9314d6f9799f5bfba510cc7bcd468c01,0dacf04c5ad59fd5a0cc1faa07c34e39,166.99,21.66,188.65,credit_card,"GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...",2
1,373cbeecea8286a2b66c97b1b157ec46,583174fbe37d3d5f0d6661be3aad1786,89.90,9.38,99.28,credit_card,Péssimo,1
2,2c5e27fc178bde7ac173c9c62c31b070,0ce9a24111d850192a933fcaab6fbad3,159.90,28.50,188.40,credit_card,Não gostei ! Comprei gato por lebre,1
3,58044bca115705a48fe0e00a21390c54,68e55ca79d04a79f20d4bfc0146f4b66,137.00,17.67,154.67,credit_card,Sempre compro pela Internet e a entrega ocorre...,1
4,9fd59cd04b42f600df9f25e54082a8d1,3c314f50bc654f3c4e317b055681dff9,59.00,31.70,296.89,credit_card,Nada de chegar o meu pedido.,1
...,...,...,...,...,...,...,...,...
19456,51de4e06a6b701cb2be47ea0e689437b,b7467ae483dbe956fe9acdf0b1e6e3f4,105.00,11.62,699.72,credit_card,Bom dia \r\nDas 6 unidades compradas só recebi...,3
19457,2ee221b28e5b6fceffac59487ed39348,f2d12dd37eaef72ed7b1186b2edefbcd,48.90,7.87,56.77,credit_card,Foto muito diferente principalmente a graninha...,2
19458,5085bc489aa6b58a29c4f922d59ff826,18ed848509774f56cc8c1c0a1903ad7f,49.99,15.10,65.09,credit_card,Tive um problema na entrega em que o correio c...,2
19459,a709d176f59bc3af77f4149c96bae357,d5cb12269711bd1eaf7eed8fd32a7c95,29.99,12.79,42.78,credit_card,"O produto não foi enviado com NF, não existe v...",3
